# Colab · Fine-tune `whisper-small` (full or LoRA) on Common Voice 27.0 `tr`

Runs `src.training.train_whisper`, then `src.evaluation.evaluate_asr` on the full test split.
All logic lives in the repository; this notebook only sets up the runtime. Details:
`docs/training.md`.

**Before running**

1. *Runtime → Change runtime type → T4 GPU.*
2. On your computer: `python scripts/pack_data.py --splits train dev` and upload
   `data/colab/cv-corpus-27.0-2026-09-11-train-dev.tar` to Google Drive (`DRIVE_DIR` below),
   next to the test archive used for the baseline.
3. Set `METHOD` below (`"lora"` first is suggested: smaller checkpoints), then *Run all*.

**Free tier:** sessions can end after a few hours or when the tab is idle. Checkpoints go to
Drive every 500 steps; after a disconnect just *Run all* again and training resumes. A
finished run is detected and not trained again.

In [ ]:
METHOD = "lora"  # "lora" or "full"

REPO_URL = "https://github.com/MuhammedTahaBesikoglu/turkish-asr-summarization.git"
BRANCH = "main"
DRIVE_DIR = "/content/drive/MyDrive/turkish-asr"  # data archives; outputs, mlflow, results go here
DATA_TARS = [
    "cv-corpus-27.0-2026-09-11-train-dev.tar",
    "cv-corpus-27.0-2026-09-11-test.tar",
]
REPO_DIR = "/content/turkish-asr-summarization"

assert METHOD in ("lora", "full")
RUN = f"whisper-small-{METHOD}"
TRAIN_CONFIG = f"configs/train_{METHOD}.yaml"
EVAL_CONFIG = f"configs/eval_{METHOD}.yaml"
COLAB_DATA = "--data-config configs/data_colab.yaml"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!python --version
!nproc

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## Code and dependencies

Colab's preinstalled PyTorch is kept; the other libraries are installed at the versions
pinned in `requirements.txt`. Colab's preinstalled `torchao` (0.10) is removed: the project
does not use it, and peft 0.21 refuses to create LoRA layers when a torchao older than 0.16
is installed.

In [ ]:
import os

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -1
PKGS = "transformers|accelerate|peft|mlflow|jiwer|soundfile|librosa|pyyaml"
!pip install -q $(grep -E '^({PKGS})==' requirements.txt)
!pip install -q --no-deps -e .
!pip uninstall -q -y torchao
!python scripts/check_env.py

## Data and output folder

Archives are extracted to the VM's local disk (`/content/data`, as in
`configs/data_colab.yaml`); reading clips from Drive directly is slow. `outputs/` is linked to
Drive so checkpoints and the final model survive a disconnect.

In [ ]:
CV_ROOT = "/content/data/cv-corpus-27.0-2026-09-11/tr"
!mkdir -p /content/data
for tar in DATA_TARS:
    marker = f"/content/data/.{tar}.done"
    if not os.path.exists(marker):
        !tar -xf "{DRIVE_DIR}/{tar}" -C /content/data && touch "{marker}"
!ls {CV_ROOT}
!ls {CV_ROOT}/clips | wc -l

!mkdir -p "{DRIVE_DIR}/outputs"
!ln -sfn "{DRIVE_DIR}/outputs" outputs
!ls -la outputs/

## Smoke run (20 steps) and time projection

Same config, 20 steps, 64 dev clips, no MLflow, written to `/content/smoke`. The projection
is rough (it includes start-up and one checkpoint save); use it to check that `max_steps`
fits the free tier before the real run.

In [ ]:
import json
from pathlib import Path

import yaml

SMOKE = "/content/smoke"
SMOKE_STEPS, SMOKE_EVAL = 20, 64
!rm -rf {SMOKE}
!python -m src.training.train_whisper --config {TRAIN_CONFIG} {COLAB_DATA} \
    --set tracking=null --set training.output_dir={SMOKE}/out \
    --set output.results_dir={SMOKE}/results --set training.max_steps={SMOKE_STEPS} \
    --set training.eval_steps={SMOKE_STEPS} --set training.save_steps={SMOKE_STEPS} \
    --set training.logging_steps=5 --set data.eval_samples={SMOKE_EVAL}
# _exit_code is set by IPython after each ! command
assert _exit_code == 0, "Smoke run failed; see the traceback above"  # noqa: F821

smoke = json.loads(Path(f"{SMOKE}/results/train_summary.json").read_text())["resources"]
cfg = yaml.safe_load(Path(TRAIN_CONFIG).read_text())
steps, eval_steps, eval_n = (
    cfg["training"]["max_steps"],
    cfg["training"]["eval_steps"],
    cfg["data"]["eval_samples"],
)
sec_per_step = smoke["train_seconds"] / SMOKE_STEPS
sec_per_eval = smoke["eval_seconds"] * eval_n / SMOKE_EVAL
total_h = (steps * sec_per_step + steps // eval_steps * sec_per_eval) / 3600
print(
    f"{sec_per_step:.2f} s/step, {sec_per_eval / 60:.1f} min per dev evaluation, "
    f"peak GPU memory {smoke['peak_gpu_memory_gib']} GiB"
)
print(f"Projected {RUN}: {steps} steps ≈ {total_h:.1f} h including evaluations")

## Training

`--resume` continues from the last checkpoint on Drive, or starts a new run if there is
none. The MLflow database is restored from and backed up to `DRIVE_DIR/mlflow`.

In [ ]:
!python -m src.training.train_whisper --config {TRAIN_CONFIG} {COLAB_DATA} \
    --tracking-backup "{DRIVE_DIR}/mlflow" --resume
# _exit_code is set by IPython after each ! command
assert _exit_code == 0, "Training failed or was interrupted; see the output above"  # noqa: F821

## Test evaluation

The best checkpoint (`outputs/{RUN}/final`, LoRA merged) on the full Common Voice test
split, with the same script and settings as the zero-shot baseline.

In [ ]:
!python -m src.evaluation.evaluate_asr --config {EVAL_CONFIG} {COLAB_DATA}
# _exit_code is set by IPython after each ! command
assert _exit_code == 0, "Test evaluation failed; see the traceback above"  # noqa: F821

## Save results to Drive

In [ ]:
import shutil

out = Path(f"results/finetune/{RUN}")
shutil.copy(f"outputs/{RUN}/train_summary.json", out / "train_summary.json")
report = json.loads((out / "metrics.json").read_text(encoding="utf-8"))
assert report["is_full_split"], "metrics.json is not from the full test split"
train = json.loads((out / "train_summary.json").read_text(encoding="utf-8"))
print(json.dumps({k: report[k] for k in ["model", "num_clips", "metrics"]}, indent=2))
print(
    json.dumps(
        {
            k: train[k]
            for k in ["method", "parameters", "global_step", "best_dev_metrics", "resources"]
        },
        indent=2,
    )
)

!mkdir -p "{DRIVE_DIR}/results/finetune"
!cp -r {out} "{DRIVE_DIR}/results/finetune/"
!ls -la "{DRIVE_DIR}/results/finetune/{RUN}"